# Class 05 — Data Splitting Techniques

## Class Agenda

1. Why splitting matters and the leakage failure mode
2. Train / Validation / Test split — sizes, stratification
3. K-Fold Cross-Validation — derivation and variance analysis
4. Stratified, Group, and TimeSeries K-Fold
5. Nested CV for unbiased hyper-parameter search
6. Production splitting: temporal holdouts and rolling backtests
7. Mini case study on a real dataset

## Why This Topic Matters

**Importance in Machine Learning**  
All generalization claims rest on the split. A bad split inflates metrics by 10–40 pp and silently ships broken models.

**Importance in AI Systems**  
Modern AI systems (RAG, recommenders, LLM fine-tunes) all use held-out sets — temporal, user-group, or topic — to estimate true risk.

**Importance in Production ML**  
Production splits must mirror the prediction-time distribution: future timestamps, unseen users, unseen items.

**Importance in Interviews**  
FAANG interviews probe leakage, stratification, group leakage, and temporal CV almost every loop.

**Real-World Relevance**  
Uber forecasting uses rolling-origin CV; Netflix uses user-grouped CV; banks use out-of-time validation for credit models.

## Learning Outcomes

By the end of this class, learners will be able to:

- Justify the train/val/test partition mathematically
- Compute the bias–variance trade-off of K choice
- Distinguish KFold, StratifiedKFold, GroupKFold, TimeSeriesSplit
- Detect target, group, and temporal leakage
- Implement nested CV for unbiased model selection
- Use `cross_validate` with multiple scorers
- Design rolling-origin backtests for time-series
- Quantify CV variance via confidence intervals
- Build a leakage-free preprocessing + CV pipeline
- Pick a splitting strategy from data characteristics
- Diagnose distribution shift between splits

## 1. The Splitting Problem — Formal Setup

Given i.i.d. samples $\mathcal{D}=\{(x_i,y_i)\}_{i=1}^n$ drawn from $P(X,Y)$, we want an estimator of the true risk

$$R(f)=\mathbb{E}_{(X,Y)\sim P}[\ell(f(X),Y)].$$

The empirical risk on the *training* set is optimistically biased; we need a **held-out** estimator. The split $\mathcal{D}=\mathcal{D}_\text{tr}\cup\mathcal{D}_\text{val}\cup\mathcal{D}_\text{te}$ gives:

- $\mathcal{D}_\text{tr}$: fit parameters
- $\mathcal{D}_\text{val}$: choose hyper-parameters / model family
- $\mathcal{D}_\text{te}$: **one-shot** unbiased estimate of $R$

**Example.** With $n=10{,}000$, a typical split is 70/15/15. If you tune on test you re-introduce optimism roughly equal to $\sqrt{2\log H / n_\text{te}}$ for $H$ candidate models — for $H{=}1000$, $n_\text{te}{=}1500$ that's ≈ 0.10 in 0/1 loss.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
X, y = load_breast_cancer(return_X_y=True)
X_tmp, X_te, y_tmp, y_te = train_test_split(X, y, test_size=0.15, stratify=y, random_state=0)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.1765, stratify=y_tmp, random_state=0)
print({'train':len(y_tr), 'val':len(y_va), 'test':len(y_te), 'pos_rate_train':y_tr.mean(), 'pos_rate_test':y_te.mean()})

{'train': 397, 'val': 86, 'test': 86, 'pos_rate_train': np.float64(0.6272040302267002), 'pos_rate_test': np.float64(0.627906976744186)}


## 2. K-Fold Cross-Validation

Partition $\mathcal D$ into $K$ disjoint folds $\mathcal F_1,\dots,\mathcal F_K$. For each $k$ train on $\mathcal D\setminus\mathcal F_k$, evaluate on $\mathcal F_k$:

$$\widehat{R}_\text{CV}=\frac1K\sum_{k=1}^K\frac1{|\mathcal F_k|}\sum_{(x,y)\in\mathcal F_k}\ell(f_{-k}(x),y).$$

**Bias–variance of K**: small $K$ (e.g. 2) → high bias (each fold trains on half the data) and low variance; large $K$ (e.g. LOOCV with $K{=}n$) → near-zero bias but very high variance because the $n$ trained models are highly correlated. **K=5 or 10** is the practical sweet spot.

**Example.** For $n{=}1000$, 5-fold means 5 trainings on 800, eval on 200. If observed scores per fold are $\{0.91,0.93,0.88,0.92,0.90\}$, the CV estimate is $\bar s = 0.908$ with std $\approx 0.018$ — report $\bar s \pm 1.96\cdot \text{std}/\sqrt5 = 0.908\pm 0.016$.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
pipe = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=2000))])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
res = cross_validate(pipe, X, y, cv=cv, scoring=['accuracy','roc_auc'], return_train_score=True)
import numpy as np
print('val_acc', res['test_accuracy'].round(3), 'mean', res['test_accuracy'].mean().round(3),
      'CI±', (1.96*res['test_accuracy'].std()/np.sqrt(5)).round(3))

val_acc [0.956 0.974 0.982 1.    0.982] mean 0.979 CI± 0.012


## 3. Stratified, Group, and Time-Series K-Fold

- **StratifiedKFold** preserves class proportions per fold — mandatory for imbalanced classification.
- **GroupKFold** ensures the same group (user_id, patient_id) never appears in both train and val — prevents *group leakage*.
- **TimeSeriesSplit** uses an expanding window: fold $k$ trains on the first $k\cdot s$ observations and validates on the next $s$. **Never shuffle time series.**

**Example.** Credit-card fraud: 10 transactions per user, 100 users. Naive KFold mixes a user's transactions across folds; the model memorises user-level patterns and val AUC is artificially 0.99. With GroupKFold(groups=user_id) val AUC drops to a realistic 0.84.

In [ ]:
from sklearn.model_selection import GroupKFold, TimeSeriesSplit
import numpy as np, pandas as pd
rng = np.random.default_rng(0)
groups = np.repeat(np.arange(20), 50)   # 20 users × 50 rows
Xg = rng.normal(size=(1000, 4)); yg = (Xg.sum(1) + rng.normal(size=1000) > 0).astype(int)
gk = GroupKFold(n_splits=5)
for k,(tr,va) in enumerate(gk.split(Xg, yg, groups)):
    print(f'fold{k} train_groups={np.unique(groups[tr]).size}  val_groups={np.unique(groups[va]).size}  overlap={len(np.intersect1d(groups[tr],groups[va]))}')
print('--- TimeSeriesSplit ---')
for k,(tr,va) in enumerate(TimeSeriesSplit(n_splits=4).split(np.arange(20))):
    print(f'fold{k} train={tr.tolist()} val={va.tolist()}')

fold0 train_groups=16  val_groups=4  overlap=0
fold1 train_groups=16  val_groups=4  overlap=0
fold2 train_groups=16  val_groups=4  overlap=0
fold3 train_groups=16  val_groups=4  overlap=0
fold4 train_groups=16  val_groups=4  overlap=0
--- TimeSeriesSplit ---
fold0 train=[0, 1, 2, 3] val=[4, 5, 6, 7]
fold1 train=[0, 1, 2, 3, 4, 5, 6, 7] val=[8, 9, 10, 11]
fold2 train=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] val=[12, 13, 14, 15]
fold3 train=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] val=[16, 17, 18, 19]


## 4. Nested Cross-Validation

Tuning hyper-parameters on the **same** CV used to report scores leaks the validation set into model selection and overstates generalisation. Nested CV separates the two:

- **Outer loop** ($K_\text{out}$ folds): unbiased risk estimate.
- **Inner loop** ($K_\text{in}$ folds inside each outer training fold): hyper-parameter search.

Cost is $K_\text{out}\cdot K_\text{in}\cdot|\text{grid}|$ fits.

**Example.** Outer=5, Inner=3, grid of 20 → 300 model fits per algorithm. For expensive models, switch inner to a single 80/20 split (Hold-out + Outer-CV).

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.svm import SVC
inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)
gs = GridSearchCV(SVC(), {'C':[0.1,1,10],'gamma':['scale','auto']}, cv=inner, n_jobs=-1)
nested = cross_val_score(gs, X, y, cv=outer, scoring='roc_auc', n_jobs=-1)
print('nested AUC', nested.round(3), 'mean', nested.mean().round(3))

nested AUC [0.961 0.982 0.971 0.992 0.987] mean 0.979


## 5. Production Splitting — Temporal & Rolling

For online systems the only honest evaluation is **out-of-time (OOT)**: train on data before time $t$, score on $[t, t+\Delta]$. Rolling-origin backtests repeat this for many $t$. Banks call this **vintage analysis**; Uber calls it **backtesting**.

**Example.** A demand-forecast model trained on Jan–Sep, validated on Oct, retrained Jan–Oct, validated on Nov, etc. The mean MAPE across the rolling windows is the production estimate.

## Tradeoffs & Pitfalls

- **Tuning on test** inflates reported metrics — touch test once.
- **Shuffling time series** leaks the future.
- **Per-row split with grouped data** leaks identity (user/patient/session).
- **Stratification on tiny classes** fails when $n_\text{class} < K$.
- **CV mean without CI** hides variance — always report $\pm 1.96\sigma/\sqrt K$.

## Production Considerations
Maintain a frozen golden test set, version it (DVC/lakefs), recompute drift between training and serving distributions, monitor population stability index (PSI) per feature, and re-split when drift breaches threshold.

## Mini Case Study — Stratified vs naive split on imbalanced data

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
Xi, yi = make_classification(n_samples=5000, weights=[0.97,0.03], random_state=0)
for strat in [None, yi]:
    Xtr,Xte,ytr,yte = train_test_split(Xi, yi, test_size=0.2, stratify=strat, random_state=0)
    m = LogisticRegression(max_iter=2000, class_weight='balanced').fit(Xtr, ytr)
    print('stratified=', strat is not None, 'test pos rate=', yte.mean().round(4),
          'AUC=', roc_auc_score(yte, m.predict_proba(Xte)[:,1]).round(3))

stratified= False test pos rate= 0.041 AUC= 0.938
stratified= True test pos rate= 0.035 AUC= 0.934
